# Adaptive classifier
https://github.com/codelion/adaptive-classifier/tree/main

In [ ]:
# %pip install pandas scikit-learn stopwordsiso datasets ipywidgets xgboost mlflow hf_xet tiktoken sentencepiece adaptive-classifier 

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter
import stopwordsiso as stopwords
from datasets import load_dataset
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import mlflow.sklearn

tqdm.pandas()

In [ ]:
# configs
data_path = "../datasets/dataset/full_text_dataset.csv"
ml_flow_url = "https://mlflow.com/"
ml_flow_exp_name = "adaptive-classifier"

### Setup experiment and MlFlow things

In [ ]:
# Setup experiment and MlFlow things
mlflow.set_tracking_uri(ml_flow_url)
mlflow.set_experiment(ml_flow_exp_name)

### Create stopwords

In [ ]:
nlp_stopwords = stopwords.stopwords("sv")
nlp_stopwords = set(nlp_stopwords)

# Read custom stopwords from a file
with open("custom_stopwords.txt", "r", encoding="utf-8") as f:
    custom_stopwords = set(line.strip() for line in f if line.strip())

# Combine both sets
stop_words = nlp_stopwords.union(custom_stopwords)
# stop_words

### Load dataset

In [ ]:
df = pd.read_csv(data_path)
len(df)

In [ ]:
df = df.dropna()
len(df)

### Data Preparation Using New Columns to Retain Original Data

In [ ]:
# Initialize columns for labels to keep source data
df['label'] = df['class']
df['label'] = df['class']
df.head()

### Prepare data

In [ ]:
# remove rows with that is not
df = df[df['class'] != 'THIS']
len(df)

In [ ]:
# Remove rows where 'class' is 'NEW' and 'page' is 1
df = df[~((df['class'] == 'THAT') & (df['pagenr'] == 1))]
len(df)

In [ ]:
# Set the 'class' column to 'INNER' for all rows where the 'BIO' column is 'INNER'
df.loc[df['BIO'] == 'INNER', 'class'] = 'INNER'

In [ ]:
# Count the number of occurrences of each unique value in the 'class' column
df['class'].value_counts()

In [ ]:
# Your list of allowed values
keep_classes = [
    "WHEN",
    "THIS",
    "THAT",
    "BORING"
]
# keep_classes = [
#     "THIS"
# ]

# # Assuming your DataFrame is called df and the column is named 'class'
# df['class'] = np.where(df['class'].isin(keep_classes), df['class'], 'OTHER')
# df['class'].value_counts()

In [ ]:
# Balance the dataset so that each class has at most 5000 examples (randomly selected)
nMax = 50000
df = (
    df.groupby('class', group_keys=False, observed=True)
      .apply(lambda x: x.sample(n=min(nMax, len(x)), random_state=42))
      .reset_index(drop=True)
)
df['class'].value_counts()

In [ ]:
def preprocess_text(text):
    """
    Preprocess the text by removing dates, numbers, and other unwanted characters.
    
    Args:
        text (str): Input text to preprocess
        
    Returns:
        str: Preprocessed text
    """
    if not isinstance(text, str):
        return []
    
    # # Convert to lowercase
    # text = text.lower()
    
    # # Remove dates (various formats)
    # text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '', text)
    # text = re.sub(r'\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b', '', text)
    
    # # Remove all numbers
    # text = re.sub(r'\b\d+\b', '', text)
    
    # # Remove punctuation
    # text = re.sub(r'[^\w\s]', '', text)
    
    # # Remove extra whitespace
    # text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and filter stopwords
    tokens = text.split()
    # tokens = [word for word in tokens if word not in stop_words]
    
    return tokens

### Clean up text

In [ ]:
# Add tokens, apply preprocessing to the text column
df['tokens'] = df['text'].progress_apply(preprocess_text)
# Join tokens into string for each row
df['text_joined'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))
df['first_50_joined'] = df['tokens'].apply(lambda tokens: ' '.join(tokens[:50]))

### Convert string labels to integers

In [ ]:
# Initialize the LabelEncoder
le = LabelEncoder()

# This assigns a unique integer to each unique class (e.g., 'bird' → 0, 'cat' → 1, 'dog' → 2)
df['label_ids'] = le.fit_transform(df['class'])
df.head()

### Split dataset

In [ ]:
# Split based on the 'data_split' column
column_split = 'data_split'
column_split = 'type'
test_df = df[df[column_split] == 'test']
# Filter rows where 'data_split' is either 'train' or 'val'
train_df = df[df[column_split].isin(['train', 'val'])]
test_df = df[df[column_split] == 'train']


# features
column_text = 'first_50_joined'
column_text = 'text_joined'
# column_text = 'text'
X_train_raw = train_df[column_text]
X_test_raw = test_df[column_text]

# Target labels
column_label = 'class'
column_label = 'label_ids'
y_train = train_df[column_label]
y_test = test_df[column_label]

### Create DataLoader to batch the data

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Custom collate function to prevent PyTorch DataLoader from converting raw strings into tensors.
def text_collate_fn(batch):
    texts, labels = zip(*batch)
    return list(texts), list(labels)

# Custom Dataset to hold raw text and labels
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()  # Convert Series to list
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Wrap the data in a PyTorch Dataset
train_dataset = TextDataset(X_train_raw, y_train)

# Use a DataLoader to:
# - automatically batch the data
# - shuffle the dataset each epoch (for better training)
# - allow parallel loading via multiple workers (optional)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, collate_fn=text_collate_fn)

### Modelling

In [ ]:
import time
from adaptive_classifier import AdaptiveClassifier

In [ ]:
# helper to download model from huggingface

# from transformers import BigBirdModel, BigBirdTokenizer
# model = BigBirdModel.from_pretrained("google/bigbird-roberta-large")
# tokenizer = BigBirdTokenizer.from_pretrained("google/bigbird-roberta-large")

In [ ]:
# Initialize with any HuggingFace model
classifier = AdaptiveClassifier("bert-base-uncased")
classifier = AdaptiveClassifier("google/bigbird-roberta-large")

In [ ]:
# https://colab.research.google.com/drive/1VyxVubB8LXXES6qElEYJL241emkV_Wxc?usp=sharing#scrollTo=hezCEDh9Sa4k

In [ ]:
def train_in_chunks(classifier, texts, labels, chunk_size=50):
    """Train the classifier in smaller chunks"""
    for i in range(0, len(texts), chunk_size):
        chunk_texts = texts[i:i + chunk_size]
        chunk_labels = labels[i:i + chunk_size]

        # Add examples in this chunk
        classifier.add_examples(chunk_texts, chunk_labels)

        # # Clear memory
        # gc.collect()
        # if torch.cuda.is_available():
        #     torch.cuda.empty_cache()

        print(f"Processed chunk {i//chunk_size + 1}/{(len(texts)-1)//chunk_size + 1}")

In [ ]:
# 2. Train base model
print("\nTraining base model...")
train_in_chunks(classifier, X_train_raw.tolist(), y_train.tolist())

In [ ]:
# Save the classifier
print("\nSaving classifier...")
classifier.save("adaptive_classifier")

In [ ]:
def evaluate_efficiently(classifier, texts, labels, batch_size=100):
    """Memory-efficient evaluation"""
    correct = 0
    total = len(texts)

    for i in range(0, total, batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_labels = labels[i:i + batch_size]

        for text, true_label in zip(batch_texts, batch_labels):
            pred = classifier.predict(text, k=1)[0]
            if pred[0] == true_label:
                correct += 1

        # # Clear memory
        # gc.collect()
        # if torch.cuda.is_available():
        #     torch.cuda.empty_cache()

    return correct / total

In [ ]:
# 3. Evaluate
print("\nEvaluating...")
accuracy = evaluate_efficiently(classifier, X_test_raw.tolist(), y_test.tolist())

In [ ]:
accuracy